# Mini-GPT — Data Prep: OpenWebText (~100MB) + GPT-2 BPE

Streams a ~100MB slice of **OpenWebText** (no need to download the full ~38GB dataset),
tokenizes it with **GPT-2's BPE tokenizer** via `tiktoken`, and saves train/val splits as
raw binary token files (`train.bin` / `val.bin`) that `train.py` can memory-map directly.

This only needs to be run **once** — after this, every training run (across all 6 phases)
reuses the same `train.bin` / `val.bin` files instead of re-downloading or re-tokenizing.

**Note:** this needs internet access, which Colab has by default — no GPU needed for this
particular notebook, just a normal CPU runtime is fine.

## Setup

In [ ]:
!pip install -q tiktoken datasets

In [ ]:
import os

import numpy as np
import tiktoken
from datasets import load_dataset

## Config

`TARGET_SIZE_MB` controls how much raw text we pull before stopping — keep this at 100
unless you specifically want a bigger/smaller slice (bigger = slower per-phase training
runs, smaller = less realistic results).

In [ ]:
TARGET_SIZE_MB = 100          # stop streaming once raw text hits ~this size
VAL_FRACTION = 0.1            # fraction of tokens held out for validation
ENCODING_NAME = 'gpt2'        # tiktoken's GPT-2 BPE encoding -> vocab_size = 50257
OUTPUT_DIR = '/content'       # save train.bin / val.bin here (or point at your Drive)

TARGET_SIZE_BYTES = TARGET_SIZE_MB * 1024 * 1024

## Step 1 — Stream OpenWebText until we hit ~100MB

`streaming=True` means we never download the full ~38GB dataset — documents are pulled
one at a time over the network, and we stop as soon as we've collected enough.

In [ ]:
def collect_text(target_bytes):
    print(f'Streaming OpenWebText until we hit ~{target_bytes / 1024 / 1024:.0f} MB of raw text...')

    dataset = load_dataset(
        'Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True
    )

    chunks = []
    total_bytes = 0
    n_docs = 0

    for example in dataset:
        text = example['text']
        chunks.append(text)
        total_bytes += len(text.encode('utf-8'))
        n_docs += 1

        if n_docs % 500 == 0:
            print(f'  ...{n_docs} docs, {total_bytes / 1024 / 1024:.1f} MB so far')

        if total_bytes >= target_bytes:
            break

    full_text = '\n\n'.join(chunks)
    actual_mb = len(full_text.encode('utf-8')) / 1024 / 1024
    print(f'Done collecting: {n_docs} documents, {actual_mb:.1f} MB raw text')
    return full_text

## Step 2 — Tokenize with GPT-2 BPE

`encode_ordinary` skips special-token handling (like `<|endoftext|>`) — we don't need that
machinery for a raw training corpus, just plain token ids.

In [ ]:
def tokenize(text):
    enc = tiktoken.get_encoding(ENCODING_NAME)
    print(f"Tokenizing with '{ENCODING_NAME}' BPE (vocab_size={enc.n_vocab})...")

    ids = enc.encode_ordinary(text)
    print(f'Total tokens: {len(ids):,}')
    return ids, enc.n_vocab

## Step 3 — Split train/val and save as binary files

`uint16` is enough headroom for GPT-2's vocab (50257 < 65536) and is half the size of the
default `int64` numpy would otherwise use — matters once you're memory-mapping this file
repeatedly during training.

In [ ]:
def save_splits(ids, val_fraction, output_dir):
    ids = np.array(ids, dtype=np.uint16)

    n_val = int(len(ids) * val_fraction)
    train_ids = ids[:-n_val]
    val_ids = ids[-n_val:]

    train_path = os.path.join(output_dir, 'train.bin')
    val_path = os.path.join(output_dir, 'val.bin')

    train_ids.tofile(train_path)
    val_ids.tofile(val_path)

    print(f'train.bin: {len(train_ids):,} tokens ({os.path.getsize(train_path) / 1024 / 1024:.1f} MB)')
    print(f'val.bin:   {len(val_ids):,} tokens ({os.path.getsize(val_path) / 1024 / 1024:.1f} MB)')

## Run it

In [ ]:
text = collect_text(TARGET_SIZE_BYTES)
ids, vocab_size = tokenize(text)
save_splits(ids, VAL_FRACTION, OUTPUT_DIR)

print()
print('Done. In model/gpt.py\'s GPTConfig, set:')
print(f'    vocab_size = {vocab_size}')

## (Optional) Save to Google Drive

`/content` is wiped when your Colab session ends. If you don't want to re-run the
streaming step next session, mount Drive and copy `train.bin` / `val.bin` there.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/train.bin /content/val.bin /content/drive/MyDrive/mini-gpt-project/data/

## Next up

Next file: `train.py` — the training loop for Phase 1 (baseline) through Phase 3
(memory engineering), reading `train.bin` / `val.bin` produced here.